In [4]:
import requests
import pandas as pd
import ast

from keys import TMDB_API_KEY

BASE_URL = "https://api.themoviedb.org/3"

# Load genre lookup without hardcoding
df_genres = pd.read_csv("genres.csv") # Listed id, genre name
genre_lookup = dict(zip(df_genres["id"], df_genres["name"])) # Combine ID and word into a list


In [5]:
def search_movie(term):
    url = f"{BASE_URL}/search/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "query": term
    }
    response = requests.get(url, params=params)

    # Check to make sure results are actually being sent
    if response.status_code != 200:
        print("Error: ", response.status_code)
        print(response.text)
        return []


    data = response.json()

    # If something is returned that isn't expected
    if "results" not in data or data["results"] is None:
        print("TMDB returned no results.")
        print(data)
        return []
    
    results = data.get("results", [])

    movies = []
    for m in results:
        movies.append({
            "id": m["id"],
            "title": m["title"],
            "year": (m.get("release_date") or "")[:4],
            "overview": m.get("overview", ""),
            "genre_ids": m.get("genre_ids", [])
        })



    return movies

In [14]:
# Testing Movie lookup without hardcoding
term = input("Search for a movie: ")
movies = search_movie(term)
print(movies[:3])

[{'id': 157336, 'title': 'Interstellar', 'year': '2014', 'overview': 'The adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the vast distances involved in an interstellar voyage.', 'genre_ids': [12, 18, 878]}, {'id': 529107, 'title': "Inside 'Interstellar'", 'year': '2015', 'overview': 'Cast and crew of Christopher Nolan\'s "Interstellar" discuss project origins, the film\'s imagery, ambitions, incorporating IMAX footage, the human element within the film, arm shooting locations outside of Calgary, the set construction and design, working with real corn, mechanical characters, including backstory, design, the blend of practical and digital effects in bringing them to life, the differences in the characters, the human performances behind the characters, the creative process behind the film\'s music, Icelandic locations, vehicle interiors, the processes of simulating the absence of gravity, the cruc

In [6]:
def submit_review(movie_id, rating, genre_ids=None):
    # Validate User review
    if rating < 1 or rating > 5:
        print("You must pick between 1 and 5 stars.")
        return
    
    # Load existing reviews (if any)
    try:
        df = pd.read_csv("reviews.csv")
    except FileNotFoundError:
        df = pd.DataFrame(columns=["movie_id", "rating", "genre_ids"])

    # Add a new row for the new review
    new_row = {
        "movie_id": movie_id,
        "rating": rating,
        "genre_ids": str(genre_ids) if genre_ids is not None else "" # Add empty string if no genre id string to keep csv clean
    }
    # Add new review to the bottom of reviews.csv
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    # Save new review to reviews.csv
    df.to_csv("reviews.csv", index=False)
    print("Your review was saved!")

In [25]:
# Test submit_review without hardcoding
term = input("Search for a movie: ")
movies = search_movie(term)

# Show search results
for i, m in enumerate(movies):
    print(f"{i+1}. {m['title']} ({m['year']})")

# User selects one from list of results
choice = int(input("Select a movie from the list using the number: "))
selected = movies[choice]

# User enters rating
rating = int(input("Enter your star review (1-5): "))

# Submit review
submit_review(selected["id"], rating, selected["genre_ids"])


1. Star Wars (1977)
2. Star Wars: The Last Jedi (2017)
3. Star Wars: The Force Awakens (2015)
4. Star Wars: The Rise of Skywalker (2019)
5. Star Wars: Episode I - The Phantom Menace (1999)
6. Rogue One: A Star Wars Story (2016)
7. Solo: A Star Wars Story (2018)
8. Star Wars: Episode II - Attack of the Clones (2002)
9. Star Wars: The Clone Wars (2008)
10. Star Wars: Episode III - Revenge of the Sith (2005)
11. Star Wars: Starfighter (2027)
12. Star Wars Tech (2007)
13. Star Wars Spoofs (2011)
14. Saving Star Wars (2004)
15. Star Wars: Heroes & Villains (2005)
16. Star Wars: Greatest Moments (2015)
17. Robot Chicken: Star Wars (2007)
18. LEGO Star Wars Holiday Special (2020)
19. Star Wars Dreams (2003)
20. The Making of Star Wars (1977)
Your review was saved!


In [7]:
def get_movie_details(movie_id):

    # Get movie details from TMDB
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {
        "api_key": TMDB_API_KEY
    }
    response = requests.get(url, params=params)

    # Check to make sure request is fufilled by TMDB API
    if response.status_code != 200:
        print("TMDB request error:", response.status_code)
        return {}
    
    # Return details in JSON format
    return response.json()

In [ ]:
# Testing get_movie_details
term = input("Search for a movie: ")
movies = search_movie(term)

# Show results of search
for i, m in enumerate(movies):
    print(f"{i+1}. {m['title']} ({m['year']})")

# Have user pick one to view details
selected = movies[int(input("Select a movie by the number on the list: ")) - 1]
details = get_movie_details(selected["id"])

print(details)

1. The Punisher: One Last Kill (2026)
2. The Punisher (2004)
3. The Punisher (1989)
4. Punisher: War Zone (2008)
5. Punisher (1968)
6. Avengers Confidential: Black Widow & Punisher (2014)
7. Female Teacher Punisher: Goddess of Revenge (1990)
8. The Lady Punisher (1994)
9. Azami the Punisher (1998)
10. Female Teacher Punisher 2: Goddess of Hell (1990)
11. The Punisher: Dirty Laundry (2012)
12. Punisher: Hell Tickets (2008)
13. Johnny: The Punisher (2019)
14. The Courier Punisher ()
15. The Poultry Punisher (2023)
16. Female Punisher Zebra: Sexy Edition (1990)
17. Punisher: No Good Deed ()
18. Butt Attack Punisher Girl Gautaman: The Birth of Gautaman (1994)
19. Army of One: Punisher Origins (2004)
20. Keepin' It Real: 'Punisher' Stunts (2004)
{'adult': False, 'backdrop_path': '/ziexSS07RmowuyqAdJMh1sGxjHg.jpg', 'belongs_to_collection': {'id': 635362, 'name': 'The Punisher Collection', 'poster_path': '/uirltOKTtjdxw3CUSMwbvxLl0o0.jpg', 'backdrop_path': '/wy3mGT2BxqNiBb3JV6lAQwOLrM5.jpg'},

In [9]:
def movies_with_genre(df_movies, genre_id):
    return df_movies[df_movies["genre_ids"].apply(lambda g: genre_id in g)]

In [ ]:
# Test movies_with_genre() filtering
term = input("Search for a movie: ")
movies = search_movie(term)
df_movies = pd.DataFrame(movies)
print(df_movies)

# Prompt user for genre for filtering
genre_name = input("Enter a genre to filter by: ")

# Load genre lookup
df_genres = pd.read_csv("genres.csv")
df_genres.columns = df_genres.columns.str.strip()

# Convert names back to ID
genre_id = df_genres[df_genres["name"] == genre_name]["id"].iloc[0]

# Filter movies by selected genre
filtered = movies_with_genre(df_movies, genre_id)
print(filtered)

        id                        title  year  \
1  1634942  Jason Statham Stole My Bike  2027   

                                            overview genre_ids  
1  Jason Statham in the role of a lifetime, playi...  [28, 35]  


In [ ]:
def generate_movieboard(top_n=10):
    # Read reviews.csv and check to see if there is reviews
    try:
        df = pd.read_csv("reviews.csv")
    except FileNotFoundError:
        print("No reviews yet.")
        return
    
    # No reviews present?
    if df.empty:
        print("No reviews yet.")
        return
    
    # Convert genre_ids back to lists
    df["genre_ids"] = df["genre_ids"].apply(lambda x: ast.literal_eval(x) if x else []) # Add space if no genre id is present
    
    # group movies by ID
    grouped_ID = df.groupby("movie_id")["rating"].agg(["mean", "count"]).reset_index()
    grouped_ID = grouped_ID.sort_values(by=["mean", "count"], ascending=[False, False])

    # Show top movies based on set value (n)
    print("Top movies:")
    for i, row in grouped_ID.head(top_n).iterrows():
        movie_id = int(row["movie_id"])
        avg = row["mean"]
        count = int(row["count"])

        # Get movie title from TMDB 
        details = get_movie_details(movie_id)
        title = details.get("title", "Unknown title")

        # Convert TMDB Genre IDs to words
        tmdb_genres = details.get("genres", [])
        genre_names = [genre_lookup[g["id"]] for g in tmdb_genres if g["id"] in genre_lookup]

        # Print movieboard entry
        print(f"{title} ({', '.join(genre_names)}) -- Avg: {avg:.2f} from {count} reviews")
    

In [26]:
# Test generate_movieboard
generate_movieboard()

Top movies:
Idiocracy (Comedy, Science Fiction, Adventure) -- Avg:  5.00 from 1 reviews
Star Wars Dreams (TV Movie, Documentary) -- Avg:  5.00 from 1 reviews
Interstellar (Adventure, Drama, Science Fiction) -- Avg:  3.00 from 1 reviews


In [23]:
# Test csv loading
df = pd.read_csv("reviews.csv")
print(df)

   movie_id  rating      genre_ids
0      7512     NaN  [35, 878, 12]
1      7512     NaN  [35, 878, 12]
2      7512     5.0  [35, 878, 12]
3    157336     3.0  [12, 18, 878]
